# Rizzler V26 — Real Gemini End-to-End Qualification (Kaggle)

**Real provider, no mocked Work Agent outputs.** Enable Notebook **Internet**, add Kaggle Secret `GEMINI_API_KEY`, and run all cells in order. CPU is sufficient; T4 is optional. Costs may be incurred by live Gemini requests. Passing this notebook is not an independent production release certificate.


In [ ]:
import os, sys, subprocess, pathlib, hashlib, json, secrets, shutil, time
from datetime import datetime, timezone

WORK=pathlib.Path("/kaggle/working")
REPO=WORK/"rizzler-v26-live"
BRANCH="feature/v24-context-convergence-and-safe-repair"
GIT="https://github.com/SAUNAK359/Voice-AI.git"
RUN=WORK/("Rizzler_V26_Live_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"-"+secrets.token_hex(3))
RUN.mkdir(parents=True,exist_ok=True)

if not (REPO/".git").exists():
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,GIT,str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"fetch","origin",BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO),"checkout",BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO),"reset","--hard","origin/"+BRANCH],check=True)
SOURCE_SHA=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
print("Audited source:",SOURCE_SHA)
print("Work directory:",RUN)


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[work,test]","pymupdf>=1.25,<2"],check=True)
components=["libreoffice-writer","libreoffice-impress","libreoffice-calc"]
def installed(p):
    x=subprocess.run(["dpkg-query","-W","-f=${Status}",p],text=True,capture_output=True)
    return x.returncode==0 and "install ok installed" in x.stdout
missing=[x for x in components if not installed(x)]
if missing:
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq",*missing],check=True)
assert all(installed(x) for x in components)
print("Native Word, PowerPoint, PDF and Excel strict QA dependencies installed.")


In [ ]:
from kaggle_secrets import UserSecretsClient
key=UserSecretsClient().get_secret("GEMINI_API_KEY")
if not key:
    raise RuntimeError("Enable Kaggle Secret GEMINI_API_KEY for this notebook.")
env=os.environ.copy()
env["GEMINI_API_KEY"]=key
print("Gemini secret loaded without printing the credential.")


In [ ]:
command=[sys.executable,str(REPO/"scripts"/"qualify_v26_live.py"),
         "--repo",str(REPO),"--output",str(RUN),
         "--expected-sha",SOURCE_SHA]
# Run as an independent process to avoid nested asyncio.run in Jupyter.
outcome=subprocess.run(command,cwd=str(REPO),env=env,check=False)
verdict_path=RUN/"V26_QUALIFICATION.json"
if verdict_path.is_file():
    report=json.loads(verdict_path.read_text(encoding="utf-8"))
    print("Release status:",report.get("release_status"))
    print("Offline:",report.get("offline_passed"))
    print("Real Gemini multi-artifact:",report.get("live_passed"))
    print("All-new V26 features:",report.get("v26_foundation_flags"))
else:
    print("Runner stopped before the final verdict. Inspect setup output.")
print("Process code:",outcome.returncode)


In [ ]:
from IPython.display import FileLink,display
zips=sorted(WORK.glob("Rizzler_V26_Qualification_*.zip"),
            key=lambda p:p.stat().st_mtime,reverse=True)
if not zips:
    print("No qualification ZIP found. Review the preceding cell.")
else:
    candidate=zips[0]
    print("Evidence:",candidate.name,
          "| MiB:",round(candidate.stat().st_size/2**20,2))
    display(FileLink(str(candidate)))
    print("Upload this ZIP for exact root-cause and artifact inspection.")
